# 03 - Ingesta Streaming a Bronze

## Proyecto: Cloud Provider Analytics

Objetivo: implementar la ingesta de eventos JSONL
mediante Spark Structured Streaming, conservando
trazabilidad y permitiendo la recuperación del procesamiento.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
from google.colab import drive
from pathlib import Path
from pyspark.sql import SparkSession

LANDING_PATH = Path(
    "/content/drive/MyDrive/cloud-provider-analytics-data/landing"
)

BRONZE_PATH = Path(
    "/content/drive/MyDrive/cloud-provider-analytics-data/bronze"
)

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/cloud-provider-analytics-data/checkpoints"
)

spark = (
    SparkSession.builder
    .appName("CloudProviderAnalytics_StreamingBronze")
    .master("local[*]")
    .getOrCreate()
)

print("Spark:", spark.version)
print("Landing existe:", LANDING_PATH.exists())
print("Bronze:", BRONZE_PATH)
print("Checkpoints:", CHECKPOINT_PATH)


Spark: 4.0.5
Landing existe: True
Bronze: /content/drive/MyDrive/cloud-provider-analytics-data/bronze
Checkpoints: /content/drive/MyDrive/cloud-provider-analytics-data/checkpoints


In [ ]:
import json

STREAM_PATH = LANDING_PATH / "usage_events_stream"

archivos_jsonl = sorted(STREAM_PATH.glob("*.jsonl"))

print("Existe carpeta streaming:", STREAM_PATH.exists())
print("Cantidad de archivos JSONL:", len(archivos_jsonl))

if archivos_jsonl:
    print("\nPrimeros archivos:")
    for archivo in archivos_jsonl[:5]:
        print(archivo.name)

    print("\nEjemplos de eventos:")
    with open(archivos_jsonl[0], "r", encoding="utf-8") as f:
        for _ in range(3):
            linea = f.readline()
            if not linea:
                break
            print(json.dumps(json.loads(linea), indent=2))
else:
    print("No se encontraron archivos JSONL.")


Existe carpeta streaming: True
Cantidad de archivos JSONL: 120

Primeros archivos:
events_part_0000.jsonl
events_part_0001.jsonl
events_part_0002.jsonl
events_part_0003.jsonl
events_part_0004.jsonl

Ejemplos de eventos:
{
  "event_id": "evt_nwlh5boz2guf",
  "timestamp": "2025-08-17T01:55:00Z",
  "org_id": "org_cvs4f8cg",
  "resource_id": "res_i8kcmm7d",
  "service": "networking",
  "region": "sa-east",
  "metric": "requests",
  "value": null,
  "unit": "count",
  "cost_usd_increment": 0.9561,
  "schema_version": 2,
  "carbon_kg": 0
}
{
  "event_id": "evt_xozb632zko0g",
  "timestamp": "2025-07-16T11:20:00Z",
  "org_id": "org_rixa11dp",
  "resource_id": "res_nmk7yyz6",
  "service": "database",
  "region": "ap-northeast",
  "metric": "requests",
  "value": null,
  "unit": "count",
  "cost_usd_increment": 5.6521,
  "schema_version": 1
}
{
  "event_id": "evt_h69ugrfzcopy",
  "timestamp": "2025-08-18T18:52:00Z",
  "org_id": "org_0lzjjege",
  "resource_id": "res_c4t3x1hx",
  "service": "netwo

In [ ]:
from pyspark.sql import functions as F

prueba_value = (
    spark.read
    .schema("value STRING")
    .json(str(STREAM_PATH / "*.jsonl"))
    .withColumnRenamed("value", "value_raw")
    .withColumn("value_num", F.col("value_raw").try_cast("double"))
)

prueba_value.agg(
    F.count("*").alias("eventos"),
    F.count_if(F.col("value_raw").isNull()).alias("nulos_en_origen"),
    F.count("value_num").alias("valores_numericos"),
    F.count_if(
        F.col("value_raw").isNotNull() & F.col("value_num").isNull()
    ).alias("fallos_conversion"),
).show()

+-------+---------------+-----------------+-----------------+
|eventos|nulos_en_origen|valores_numericos|fallos_conversion|
+-------+---------------+-----------------+-----------------+
|  43200|            877|            42323|                0|
+-------+---------------+-----------------+-----------------+



## 2. Definición del esquema de eventos

Definimos un esquema explícito para leer los archivos JSONL
mediante Spark Structured Streaming.

El esquema contempla eventos de las versiones v1 y v2.

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    TimestampType
)

events_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("org_id", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("metric", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("cost_usd_increment", DoubleType(), True),
    StructField("schema_version", IntegerType(), True),
    StructField("carbon_kg", DoubleType(), True),
    StructField("genai_tokens", DoubleType(), True)
])

print(events_schema.simpleString())


struct<event_id:string,timestamp:timestamp,org_id:string,resource_id:string,service:string,region:string,metric:string,value:double,unit:string,cost_usd_increment:double,schema_version:int,carbon_kg:double,genai_tokens:double>


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType

esquema_lectura = StructType([
    StructField("value", StringType(), True) if campo.name == "value" else campo
    for campo in events_schema.fields
])

eventos_reprocesados = (
    spark.readStream
    .schema(esquema_lectura)
    .option("maxFilesPerTrigger", 10)
    .json(str(STREAM_PATH))
    .withColumnRenamed("value", "value_raw")
    .withColumn("value", F.col("value_raw").try_cast("double"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
    .withWatermark("timestamp", "60 days")
    .dropDuplicatesWithinWatermark(["event_id"])
)

print("¿Es streaming?", eventos_reprocesados.isStreaming)

¿Es streaming? True


In [ ]:
RUTA_BRONZE_PRUEBA = BRONZE_PATH / "usage_events_reprocesado"
RUTA_CHECKPOINT_PRUEBA = CHECKPOINT_PATH / "usage_events_bronze_reprocesado"

consulta_prueba = (
    eventos_reprocesados.writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", str(RUTA_CHECKPOINT_PRUEBA))
    .trigger(availableNow=True)
    .start(str(RUTA_BRONZE_PRUEBA))
)

consulta_prueba.awaitTermination()

print("Salida:", RUTA_BRONZE_PRUEBA)
print("Eventos escritos:", spark.read.parquet(str(RUTA_BRONZE_PRUEBA)).count())

Salida: /content/drive/MyDrive/cloud-provider-analytics-data/bronze/usage_events_reprocesado
Eventos escritos: 43200


In [ ]:
bronze_reprocesado = spark.read.parquet(str(RUTA_BRONZE_PRUEBA))

bronze_reprocesado.agg(
    F.count("*").alias("eventos"),
    F.countDistinct("event_id").alias("ids_distintos"),
    F.count_if(F.col("event_id").isNull()).alias("ids_nulos"),
    F.count_if(F.col("value").isNull()).alias("value_nulos"),
    F.count_if(
        F.col("value_raw").isNotNull() & F.col("value").isNull()
    ).alias("fallos_conversion"),
).show()

+-------+-------------+---------+-----------+-----------------+
|eventos|ids_distintos|ids_nulos|value_nulos|fallos_conversion|
+-------+-------------+---------+-----------+-----------------+
|  43200|        43200|        0|        877|                0|
+-------+-------------+---------+-----------+-----------------+



In [ ]:

events_test = (
    spark.read
    .schema(events_schema)
    .json(str(STREAM_PATH / "*.jsonl"))
)

print("Cantidad de eventos:", events_test.count())

print("\nEsquema:")
events_test.printSchema()

print("\nPrimeros eventos:")
events_test.show(5, truncate=False)

print("\nDistribución por versión:")
events_test.groupBy("schema_version").count().show()


Cantidad de eventos: 43200

Esquema:
root
 |-- event_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)


Primeros eventos:
+----------------+-------------------+------------+------------+---------+----------+----------------+-------+--------+------------------+--------------+---------+
|event_id        |timestamp          |org_id      |resource_id |service  |region    |metric          |value  |unit    |cost_usd_increment|schema_version|carbon_kg|
+----------------+-------------------+------------+------------+---------+----------+----------------+-------+--------+

## 3. Validación de la evolución del esquema

Verificamos los campos presentes en los archivos JSONL
para contemplar las versiones v1 y v2 antes de iniciar
la ingesta mediante Structured Streaming.

In [ ]:

import json
from collections import defaultdict

campos_por_version = defaultdict(set)
cantidad_por_version = defaultdict(int)

for archivo in archivos_jsonl:
    with open(archivo, "r", encoding="utf-8") as f:
        for linea in f:
            if not linea.strip():
                continue

            evento = json.loads(linea)
            version = evento.get("schema_version")

            campos_por_version[version].update(evento.keys())
            cantidad_por_version[version] += 1

for version in sorted(campos_por_version):
    print(f"\nVersión {version}")
    print("Cantidad de eventos:", cantidad_por_version[version])
    print("Campos encontrados:")

    for campo in sorted(campos_por_version[version]):
        print(" -", campo)

campos_definidos = set(events_schema.fieldNames())

todos_los_campos = set().union(
    *campos_por_version.values()
)

print("\nCampos que faltan en nuestro esquema:")
print(sorted(todos_los_campos - campos_definidos))



Versión 1
Cantidad de eventos: 10800
Campos encontrados:
 - cost_usd_increment
 - event_id
 - metric
 - org_id
 - region
 - resource_id
 - schema_version
 - service
 - timestamp
 - unit
 - value

Versión 2
Cantidad de eventos: 32400
Campos encontrados:
 - carbon_kg
 - cost_usd_increment
 - event_id
 - genai_tokens
 - metric
 - org_id
 - region
 - resource_id
 - schema_version
 - service
 - timestamp
 - unit
 - value

Campos que faltan en nuestro esquema:
[]


## 4. Ingesta con Spark Structured Streaming

Leemos los archivos JSONL mediante Structured Streaming,
utilizando un esquema explícito compatible con las
versiones v1 y v2.

In [ ]:
from pyspark.sql import functions as F

events_stream = (
    spark.readStream
    .schema(events_schema)
    .option("maxFilesPerTrigger", 10)
    .json(str(STREAM_PATH))
)

events_bronze_stream = (
    events_stream
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

print("¿Es un DataFrame de streaming?")
print(events_bronze_stream.isStreaming)

print("\nEsquema del streaming:")
events_bronze_stream.printSchema()


¿Es un DataFrame de streaming?
True

Esquema del streaming:
root
 |-- event_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- genai_tokens: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



## 5. Escritura de eventos en Bronze

Persistimos los eventos procesados mediante Structured Streaming
en formato Parquet, utilizando un checkpoint para registrar
el progreso y permitir reejecuciones.

In [ ]:
# Definimos las rutas de salida

BRONZE_EVENTS_PATH = BRONZE_PATH / "usage_events"

STREAM_CHECKPOINT_PATH = (
    CHECKPOINT_PATH / "usage_events_bronze"
)

print("Destino Bronze:", BRONZE_EVENTS_PATH)
print("Checkpoint:", STREAM_CHECKPOINT_PATH)

print("\n¿Existe Bronze?", BRONZE_PATH.exists())
print("¿Existe carpeta de checkpoints?", CHECKPOINT_PATH.exists())


Destino Bronze: /content/drive/MyDrive/cloud-provider-analytics-data/bronze/usage_events
Checkpoint: /content/drive/MyDrive/cloud-provider-analytics-data/checkpoints/usage_events_bronze

¿Existe Bronze? True
¿Existe carpeta de checkpoints? False


In [ ]:
# Crear la carpeta de checkpoints si no existe
CHECKPOINT_PATH.mkdir(parents=True, exist_ok=True)

print("¿Existe carpeta de checkpoints?", CHECKPOINT_PATH.exists())

# Comprobar permisos de escritura
archivo_prueba = CHECKPOINT_PATH / "prueba_escritura.txt"

try:
    archivo_prueba.write_text(
        "Prueba de escritura desde Google Colab",
        encoding="utf-8"
    )

    print("Permiso de escritura: OK")

    archivo_prueba.unlink()
    print("Archivo de prueba eliminado correctamente")

except Exception as e:
    print("Error de escritura:", e)


¿Existe carpeta de checkpoints? True
Permiso de escritura: OK
Archivo de prueba eliminado correctamente


## 6. Ejecución de Structured Streaming

Procesamos los archivos JSONL disponibles y los guardamos
en Bronze como Parquet, utilizando un checkpoint para
registrar el progreso de la ingesta.

In [ ]:
# Iniciar la ingesta de eventos hacia Bronze

query = (
    events_bronze_stream
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(STREAM_CHECKPOINT_PATH)
    )
    .trigger(availableNow=True)
    .start(str(BRONZE_EVENTS_PATH))
)

# Esperar hasta que termine el procesamiento
query.awaitTermination()

print("Streaming finalizado correctamente.")
print("Estado activo:", query.isActive)

# Mostrar información del último microbatch
if query.lastProgress:
    print("Último progreso:")
    print(query.lastProgress)


Streaming finalizado correctamente.
Estado activo: False
Último progreso:
{
    "id": "88253c75-2678-42e4-9de8-0e3d3e2f18af",
    "runId": "f4f65022-8480-49c9-97dd-72990d4b45f1",
    "name": null,
    "timestamp": "2026-11-04T22:36:46.174Z",
    "batchId": 11,
    "batchDuration": 2530,
    "numInputRows": 3600,
    "inputRowsPerSecond": 2660.7538802660756,
    "processedRowsPerSecond": 1422.9249011857708,
    "durationMs": {
        "addBatch": 1719,
        "commitOffsets": 367,
        "getBatch": 75,
        "latestOffset": 156,
        "queryPlanning": 38,
        "triggerExecution": 2530,
        "walCommit": 157
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[file:/content/drive/MyDrive/cloud-provider-analytics-data/landing/usage_events_stream]",
            "startOffset": {
                "logOffset": 10
            },
            "endOffset": {
                "logOffset": 11
            },
            "latestOffset": n

## 7. Validación de la ingesta en Bronze

Verificamos que los eventos procesados mediante Structured
Streaming se hayan almacenado correctamente en formato Parquet.

In [ ]:
# Leer los eventos guardados en Bronze

events_bronze = spark.read.parquet(
    str(BRONZE_EVENTS_PATH)
)

print("Total de eventos en Bronze:")
print(events_bronze.count())

print("\nDistribución por versión:")
events_bronze.groupBy("schema_version").count().show()

print("\nEsquema guardado:")
events_bronze.printSchema()

print("\nPrimeros eventos:")
events_bronze.select(
    "event_id",
    "org_id",
    "service",
    "cost_usd_increment",
    "carbon_kg",
    "genai_tokens",
    "ingest_ts",
    "source_file"
).show(5, truncate=False)


Total de eventos en Bronze:
43200

Distribución por versión:
+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1|10800|
|             2|32400|
+--------------+-----+


Esquema guardado:
root
 |-- event_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- genai_tokens: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = false)
 |-- source_file: string (nullable = false)


Primeros eventos:
+----------------+------------+----------+------------------+---------+------------+-----------------------+------------------------

## 8. Prueba de idempotencia

Verificamos que una segunda ejecución del streaming,
utilizando el mismo checkpoint, no duplique los eventos
que ya fueron procesados.

In [ ]:
# Guardar la cantidad de eventos antes de reejecutar

registros_antes = spark.read.parquet(
    str(BRONZE_EVENTS_PATH)
).count()

print("Registros antes de reejecutar:", registros_antes)


Registros antes de reejecutar: 43200


In [ ]:
# Segunda ejecución con el MISMO checkpoint

query_idempotencia = (
    events_bronze_stream
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(STREAM_CHECKPOINT_PATH)
    )
    .trigger(availableNow=True)
    .start(str(BRONZE_EVENTS_PATH))
)

query_idempotencia.awaitTermination()

print("Segunda ejecución finalizada.")
print("Estado activo:", query_idempotencia.isActive)

if query_idempotencia.lastProgress:
    print(
        "Registros procesados en el último microbatch:",
        query_idempotencia.lastProgress["numInputRows"]
    )
else:
    print("No se generaron nuevos microbatches.")


Segunda ejecución finalizada.
Estado activo: False
Registros procesados en el último microbatch: 0


In [ ]:
# Leer nuevamente Bronze

registros_despues = spark.read.parquet(
    str(BRONZE_EVENTS_PATH)
).count()

print("Registros antes:", registros_antes)
print("Registros después:", registros_despues)
print("Diferencia:", registros_despues - registros_antes)

if registros_antes == registros_despues:
    print("PRUEBA EXITOSA: no se duplicaron registros.")
else:
    print("ATENCIÓN: cambió la cantidad de registros.")


Registros antes: 43200
Registros después: 43200
Diferencia: 0
PRUEBA EXITOSA: no se duplicaron registros.


## 9. Análisis de duplicados y fechas de eventos

In [ ]:
from pyspark.sql import functions as F

# Cantidad total de eventos
total_eventos = events_bronze.count()

# Identificadores únicos
eventos_unicos = (
    events_bronze
    .select("event_id")
    .distinct()
    .count()
)

# Eventos con identificador nulo
eventos_id_nulos = (
    events_bronze
    .filter(F.col("event_id").isNull())
    .count()
)

print("Total de eventos:", total_eventos)
print("event_id distintos:", eventos_unicos)
print("event_id nulos:", eventos_id_nulos)
print("Registros repetidos por event_id:",
      total_eventos - eventos_unicos)

print("\nRango de fechas de los eventos:")

events_bronze.select(
    F.min("timestamp").alias("fecha_minima"),
    F.max("timestamp").alias("fecha_maxima")
).show(truncate=False)

print("\nEjemplos de IDs duplicados:")

events_bronze.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(10, truncate=False)


Total de eventos: 43200
event_id distintos: 43200
event_id nulos: 0
Registros repetidos por event_id: 0

Rango de fechas de los eventos:
+-------------------+-------------------+
|fecha_minima       |fecha_maxima       |
+-------------------+-------------------+
|2025-07-03 00:02:00|2025-08-31 23:58:00|
+-------------------+-------------------+


Ejemplos de IDs duplicados:
+--------+-----+
|event_id|count|
+--------+-----+
+--------+-----+



## 10. Preparación de prueba de watermark y deduplicación

Analizamos la distribución temporal de los eventos para
definir una estrategia de deduplicación por event_id
sin perder información válida.

In [ ]:
from pyspark.sql import functions as F

# Distribución de eventos por día
eventos_por_dia = (
    events_bronze
    .withColumn("fecha_evento", F.to_date("timestamp"))
    .groupBy("fecha_evento")
    .count()
    .orderBy("fecha_evento")
)

print("Cantidad de días con eventos:")
print(eventos_por_dia.count())

print("\nPrimeros 10 días:")
eventos_por_dia.show(10, truncate=False)

print("\nÚltimos 10 días:")
eventos_por_dia.orderBy(
    F.desc("fecha_evento")
).show(10, truncate=False)

print("\nEventos por mes:")
(
    events_bronze
    .groupBy(
        F.date_format("timestamp", "yyyy-MM").alias("mes")
    )
    .count()
    .orderBy("mes")
    .show()
)


Cantidad de días con eventos:
60

Primeros 10 días:
+------------+-----+
|fecha_evento|count|
+------------+-----+
|2025-07-03  |720  |
|2025-07-04  |720  |
|2025-07-05  |720  |
|2025-07-06  |720  |
|2025-07-07  |720  |
|2025-07-08  |720  |
|2025-07-09  |720  |
|2025-07-10  |720  |
|2025-07-11  |720  |
|2025-07-12  |720  |
+------------+-----+
only showing top 10 rows

Últimos 10 días:
+------------+-----+
|fecha_evento|count|
+------------+-----+
|2025-08-31  |720  |
|2025-08-30  |720  |
|2025-08-29  |720  |
|2025-08-28  |720  |
|2025-08-27  |720  |
|2025-08-26  |720  |
|2025-08-25  |720  |
|2025-08-24  |720  |
|2025-08-23  |720  |
|2025-08-22  |720  |
+------------+-----+
only showing top 10 rows

Eventos por mes:
+-------+-----+
|    mes|count|
+-------+-----+
|2025-07|20880|
|2025-08|22320|
+-------+-----+



## 11. Prueba controlada de deduplicación y watermark

Preparamos un entorno independiente para probar el
comportamiento de Spark Structured Streaming ante
eventos duplicados y eventos tardíos.

Esta prueba no modifica los datos originales de Bronze.

In [ ]:
# Rutas independientes para la prueba

TEST_BASE_PATH = (
    LANDING_PATH.parent / "streaming_test"
)

TEST_INPUT_PATH = TEST_BASE_PATH / "input"

TEST_OUTPUT_PATH = TEST_BASE_PATH / "output"

TEST_CHECKPOINT_PATH = (
    TEST_BASE_PATH / "checkpoint"
)

print("Carpeta de prueba:", TEST_BASE_PATH)
print("Entrada:", TEST_INPUT_PATH)
print("Salida:", TEST_OUTPUT_PATH)
print("Checkpoint:", TEST_CHECKPOINT_PATH)

print("\n¿Existe la carpeta de prueba?")
print(TEST_BASE_PATH.exists())


Carpeta de prueba: /content/drive/MyDrive/cloud-provider-analytics-data/streaming_test
Entrada: /content/drive/MyDrive/cloud-provider-analytics-data/streaming_test/input
Salida: /content/drive/MyDrive/cloud-provider-analytics-data/streaming_test/output
Checkpoint: /content/drive/MyDrive/cloud-provider-analytics-data/streaming_test/checkpoint

¿Existe la carpeta de prueba?
False


## 12. Creación de eventos de prueba

Creamos un conjunto pequeño de eventos simulados para
evaluar la deduplicación por event_id y el tratamiento
de eventos tardíos mediante Spark Structured Streaming.

In [ ]:
import json

# Crear las carpetas de prueba
TEST_INPUT_PATH.mkdir(parents=True, exist_ok=True)
TEST_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
TEST_CHECKPOINT_PATH.mkdir(parents=True, exist_ok=True)

# Eventos simulados
eventos_prueba = [
    {
        "event_id": "test_A",
        "timestamp": "2025-08-30T10:00:00Z",
        "org_id": "org_test",
        "service": "compute",
        "metric": "requests",
        "value": 100.0,
        "unit": "count",
        "cost_usd_increment": 2.5,
        "schema_version": 2
    },
    {
        "event_id": "test_A",
        "timestamp": "2025-08-30T10:00:00Z",
        "org_id": "org_test",
        "service": "compute",
        "metric": "requests",
        "value": 100.0,
        "unit": "count",
        "cost_usd_increment": 2.5,
        "schema_version": 2
    },
    {
        "event_id": "test_B",
        "timestamp": "2025-08-30T11:00:00Z",
        "org_id": "org_test",
        "service": "storage",
        "metric": "storage_gb_hours",
        "value": 50.0,
        "unit": "gb_hours",
        "cost_usd_increment": 1.2,
        "schema_version": 2
    }
]

# Guardar los eventos en un archivo JSONL
archivo_prueba = TEST_INPUT_PATH / "eventos_prueba_01.jsonl"

with open(archivo_prueba, "w", encoding="utf-8") as f:
    for evento in eventos_prueba:
        f.write(json.dumps(evento) + "\n")

print("Archivo creado:", archivo_prueba)
print("Cantidad de eventos:", len(eventos_prueba))
print("Carpeta de entrada existe:", TEST_INPUT_PATH.exists())


Archivo creado: /content/drive/MyDrive/cloud-provider-analytics-data/streaming_test/input/eventos_prueba_01.jsonl
Cantidad de eventos: 3
Carpeta de entrada existe: True


## 13. Deduplicación con watermark

Aplicamos un watermark de 1 día y eliminamos
eventos repetidos por event_id mediante
Spark Structured Streaming.

In [ ]:
# Leer los eventos de prueba mediante streaming

test_stream = (
    spark.readStream
    .schema(events_schema)
    .json(str(TEST_INPUT_PATH))
)

# Aplicar watermark y deduplicación
test_stream_dedup = (
    test_stream
    .withWatermark("timestamp", "1 day")
    .dropDuplicates(["event_id"])
)

print("¿Es un DataFrame de streaming?")
print(test_stream_dedup.isStreaming)

print("\nEsquema:")
test_stream_dedup.printSchema()


¿Es un DataFrame de streaming?
True

Esquema:
root
 |-- event_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- genai_tokens: double (nullable = true)



In [ ]:
# Ejecutar la prueba de deduplicación

query_test = (
    test_stream_dedup
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(TEST_CHECKPOINT_PATH)
    )
    .trigger(availableNow=True)
    .start(str(TEST_OUTPUT_PATH))
)

query_test.awaitTermination()

print("Prueba de streaming finalizada.")
print("Estado activo:", query_test.isActive)


Prueba de streaming finalizada.
Estado activo: False


In [ ]:
# Leer los resultados de la prueba

resultado_test = spark.read.parquet(
    str(TEST_OUTPUT_PATH)
)

print("Registros originales:", len(eventos_prueba))
print("Registros después de deduplicar:",
      resultado_test.count())

print("\nEventos conservados:")

resultado_test.select(
    "event_id",
    "timestamp",
    "service"
).orderBy("event_id").show(truncate=False)


Registros originales: 3
Registros después de deduplicar: 2

Eventos conservados:
+--------+-------------------+-------+
|event_id|timestamp          |service|
+--------+-------------------+-------+
|test_A  |2025-08-30 10:00:00|compute|
|test_B  |2025-08-30 11:00:00|storage|
+--------+-------------------+-------+



## 14. Prueba de eventos tardíos

Simulamos la llegada de eventos en distintos momentos
para observar el comportamiento del watermark.

In [ ]:
import json

# Evento más reciente para avanzar el tiempo observado
evento_reciente = {
    "event_id": "test_C",
    "timestamp": "2025-09-05T12:00:00Z",
    "org_id": "org_test",
    "service": "compute",
    "metric": "requests",
    "value": 200.0,
    "unit": "count",
    "cost_usd_increment": 4.0,
    "schema_version": 2
}

archivo_reciente = (
    TEST_INPUT_PATH / "eventos_prueba_02.jsonl"
)

# Evitar sobrescribir accidentalmente un archivo previo
if archivo_reciente.exists():
    print("El archivo ya existe. No se sobrescribió.")
else:
    with open(archivo_reciente, "w", encoding="utf-8") as f:
        f.write(json.dumps(evento_reciente) + "\n")

    print("Archivo creado:", archivo_reciente)

print("Fecha del evento:", evento_reciente["timestamp"])


Archivo creado: /content/drive/MyDrive/cloud-provider-analytics-data/streaming_test/input/eventos_prueba_02.jsonl
Fecha del evento: 2025-09-05T12:00:00Z


In [ ]:
# Reejecutar el streaming de prueba
# Manteniendo el mismo checkpoint

query_test_2 = (
    test_stream_dedup
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(TEST_CHECKPOINT_PATH)
    )
    .trigger(availableNow=True)
    .start(str(TEST_OUTPUT_PATH))
)

query_test_2.awaitTermination()

print("Segunda ejecución de prueba finalizada.")
print("Estado activo:", query_test_2.isActive)

if query_test_2.lastProgress:
    print(
        "Registros de entrada del último microbatch:",
        query_test_2.lastProgress["numInputRows"]
    )
    print(
        "Watermark reportado:",
        query_test_2.lastProgress.get(
            "eventTime", {}
        ).get("watermark")
    )


Segunda ejecución de prueba finalizada.
Estado activo: False
Registros de entrada del último microbatch: 0
Watermark reportado: 2025-09-04T12:00:00.000Z


In [ ]:
resultado_test_2 = spark.read.parquet(
    str(TEST_OUTPUT_PATH)
)

print("Cantidad de eventos conservados:")
print(resultado_test_2.count())

print("\nEventos almacenados:")

resultado_test_2.select(
    "event_id",
    "timestamp",
    "service"
).orderBy("event_id").show(truncate=False)


Cantidad de eventos conservados:
3

Eventos almacenados:
+--------+-------------------+-------+
|event_id|timestamp          |service|
+--------+-------------------+-------+
|test_A  |2025-08-30 10:00:00|compute|
|test_B  |2025-08-30 11:00:00|storage|
|test_C  |2025-09-05 12:00:00|compute|
+--------+-------------------+-------+



## 15. Simulación de un evento tardío

Creamos un evento con una fecha anterior al watermark
para evaluar cómo se comporta la deduplicación
ante datos que llegan fuera de tiempo.

In [ ]:
import json

# Evento con fecha anterior al watermark
evento_tardio = {
    "event_id": "test_D",
    "timestamp": "2025-08-25T10:00:00Z",
    "org_id": "org_test",
    "service": "database",
    "metric": "requests",
    "value": 75.0,
    "unit": "count",
    "cost_usd_increment": 1.8,
    "schema_version": 2
}

archivo_tardio = (
    TEST_INPUT_PATH / "eventos_prueba_03.jsonl"
)

if archivo_tardio.exists():
    print("El archivo ya existe. No se sobrescribió.")
else:
    with open(archivo_tardio, "w", encoding="utf-8") as f:
        f.write(json.dumps(evento_tardio) + "\n")

    print("Archivo creado:", archivo_tardio)

print("Fecha del evento tardío:",
      evento_tardio["timestamp"])


Archivo creado: /content/drive/MyDrive/cloud-provider-analytics-data/streaming_test/input/eventos_prueba_03.jsonl
Fecha del evento tardío: 2025-08-25T10:00:00Z


## 16. Procesamiento y validación de eventos tardíos

Reejecutamos Structured Streaming con el mismo checkpoint
para observar el comportamiento ante un evento anterior
al watermark.

In [ ]:
# Tercera ejecución de la prueba de streaming
# Mantenemos el mismo checkpoint y destino

query_test_3 = (
    test_stream_dedup
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(TEST_CHECKPOINT_PATH)
    )
    .trigger(availableNow=True)
    .start(str(TEST_OUTPUT_PATH))
)

query_test_3.awaitTermination()

print("Tercera ejecución finalizada.")
print("Estado activo:", query_test_3.isActive)

print("\nProgreso de los microbatches:")

for progreso in query_test_3.recentProgress:
    print(
        "Batch:", progreso["batchId"],
        "| Entrada:", progreso["numInputRows"],
        "| Watermark:",
        progreso.get("eventTime", {}).get("watermark"),
        "| Salida:",
        progreso.get("sink", {}).get("numOutputRows")
    )


Tercera ejecución finalizada.
Estado activo: False

Progreso de los microbatches:
Batch: 4 | Entrada: 1 | Watermark: 2025-09-04T12:00:00.000Z | Salida: -1


In [ ]:
# Leer el resultado actualizado

resultado_final_test = spark.read.parquet(
    str(TEST_OUTPUT_PATH)
)

print("Cantidad total de eventos conservados:")
print(resultado_final_test.count())

print("\nEventos almacenados:")

resultado_final_test.select(
    "event_id",
    "timestamp",
    "service"
).orderBy("event_id").show(truncate=False)

print("\n¿Se conservó el evento tardío test_D?")

cantidad_test_d = resultado_final_test.filter(
    F.col("event_id") == "test_D"
).count()

print("Apariciones de test_D:", cantidad_test_d)

if cantidad_test_d > 0:
    print("El evento tardío fue conservado.")
else:
    print("El evento tardío no aparece en la salida.")


Cantidad total de eventos conservados:
3

Eventos almacenados:
+--------+-------------------+-------+
|event_id|timestamp          |service|
+--------+-------------------+-------+
|test_A  |2025-08-30 10:00:00|compute|
|test_B  |2025-08-30 11:00:00|storage|
|test_C  |2025-09-05 12:00:00|compute|
+--------+-------------------+-------+


¿Se conservó el evento tardío test_D?
Apariciones de test_D: 0
El evento tardío no aparece en la salida.


## 17. Conclusiones de la ingesta streaming

Se implementó un proceso de ingesta de eventos JSONL utilizando **Apache Spark Structured Streaming**, con un esquema explícito compatible con las versiones v1 y v2.

### Resultados de la ingesta

- Se procesaron 120 archivos JSONL.
- Se almacenaron 43.200 eventos en formato Parquet en la capa Bronze.
- Se conservaron los campos `carbon_kg` y `genai_tokens`, presentes en la versión 2.
- Se incorporaron los metadatos `ingest_ts` y `source_file` para garantizar trazabilidad.
- Se configuró un checkpoint para registrar el progreso del procesamiento.

### Prueba de idempotencia

Se ejecutó nuevamente el streaming utilizando el mismo checkpoint.

- Registros antes: 43.200.
- Registros después: 43.200.
- Diferencia: 0.

El resultado demuestra que la reejecución no duplicó los archivos previamente procesados.

### Prueba controlada de deduplicación

Se implementó una prueba independiente utilizando `withWatermark("timestamp", "1 day")` y `dropDuplicates(["event_id"])`.

De tres registros de entrada, se conservaron dos identificadores únicos.

### Prueba controlada de eventos tardíos

Se avanzó el watermark hasta el 04/09/2025 a las 12:00 UTC y posteriormente se incorporó un evento con fecha 25/08/2025.

El evento fue recibido por el streaming, pero no apareció en la salida, resultado consistente con el descarte por antigüedad respecto del watermark.

### Limitaciones y próximos pasos

Las pruebas de deduplicación y watermark se realizaron en un entorno independiente y todavía deben integrarse o justificarse dentro del diseño del pipeline principal.

También queda pendiente definir cómo registrar o poner en cuarentena los eventos tardíos descartados, para mantener trazabilidad sobre los datos excluidos.